# Phase 5-A3｜Proposal × Orchestrator Integration Acceptance

**Notebook 版本：`PHASE5A3-INTEGRATION-v1`**　｜　Code pin `3e7aa3f`　｜　分支 `feature/phase5a3-integration`

`Hypothesis → Proposal (A2) → Request (A2 exporter) → Governance (A1) → Orchestrator (A1) → Synthetic / Archive Replay → Evaluator (4-C) → Decision → Archive`

- **Route A**：全部 `SYNTHETIC_FIXTURE`；Proposal 由 A2 產生、Request 由 A2 實際 Exporter 寫出、A1 Orchestrator（隔離測試模式）接收並跑完整生命週期。
- **Route B**：唯讀重播既有 Archive `20261107_081300_9439f8c`（0 batch；授權 1/1 不再消耗）。必須由研究負責人**自己填入** approver 與 https 核准連結，否則標記 `PENDING_USER_APPROVAL`。
- **不執行 `ResearchOnlyRunner.run()`、不重跑 SMA 5/20、不安裝 Core、不讀 Holdout、不修改 Core／授權／治理紀錄。**
- 所有邏輯在 `src/strategy_agent/phase5a/`；notebook 只是 thin runner。
- 輸出：`MyDrive/Quant_Research/strategy-research-agent/integrations/<YYYYMMDD_HHMMSS>_<pin7>/`（7 個檔案，O_EXCL 不可覆寫）。Drive 無法掛載時，雲端驗收標記 `PENDING`。

**請在全新的 Colab runtime 從 GitHub 開啟並 Run all。**

In [ ]:
# --- 1. 設定（不要修改 pin）---
NOTEBOOK_VERSION = "PHASE5A3-INTEGRATION-v1"
NOTEBOOK_PATH    = "notebooks/Phase5A3_Integration_Acceptance.ipynb"
AGENT_REPO = "https://github.com/hh4832/strategy-research-agent.git"
BRANCH     = "feature/phase5a3-integration"
PIN_SHA    = "3e7aa3f4ee1d446e9fc3db0d0090bb0cf4a79950"
DRIVE_BASE = "/content/drive/MyDrive/Quant_Research/strategy-research-agent"
REAL_RUN   = "20261107_081300_9439f8c"
REAL_ARCHIVE_MANIFEST_SHA256 = "328f2cf99c33b4d9d427327d092aaf31da619b5f5ad948c815e75b04b92345a9"
AGENT_ROOT = "/content/strategy-research-agent-phase5a3"

import hashlib, json, os, re, subprocess, sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
assert sys.version_info >= (3, 11), sys.version
CLEAN_RUNTIME = not any(m in sys.modules for m in ("strategy_agent", "market_research_core"))
AGENT_ROOT = Path(AGENT_ROOT)
def git(*a, cwd=None):
    r = subprocess.run(["git", *a], cwd=cwd, text=True, capture_output=True)
    if r.returncode: raise RuntimeError("git " + a[0] + " failed: " + r.stderr[-400:])
    return r.stdout.strip()
print(NOTEBOOK_VERSION, "| pin", PIN_SHA[:7], "| python", sys.version.split()[0], "| clean runtime:", CLEAN_RUNTIME)

In [ ]:
# --- 2. Version Sync ---
assert CLEAN_RUNTIME, "STOP: 此 runtime 已載入 strategy_agent / market_research_core；請 Runtime → Restart session 後 Run all"
if not AGENT_ROOT.exists():
    git("clone", "--quiet", AGENT_REPO, str(AGENT_ROOT)); git("checkout", "--quiet", "--detach", PIN_SHA, cwd=AGENT_ROOT)
    print("Fresh checkout created")
else:
    print("Existing checkout detected: verifying before reuse")
remote = git("remote", "get-url", "origin", cwd=AGENT_ROOT)
assert remote.rstrip("/").removesuffix(".git") == AGENT_REPO.removesuffix(".git"), "STOP: wrong remote " + remote
head = git("rev-parse", "HEAD", cwd=AGENT_ROOT)
status = git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT)
git("fetch", "--quiet", "origin", BRANCH, cwd=AGENT_ROOT)
remote_head = git("rev-parse", "FETCH_HEAD", cwd=AGENT_ROOT)
on_branch = subprocess.run(["git", "merge-base", "--is-ancestor", PIN_SHA, remote_head], cwd=AGENT_ROOT).returncode == 0
nb = subprocess.run(["git", "show", f"{remote_head}:{NOTEBOOK_PATH}"], cwd=AGENT_ROOT, text=True, capture_output=True)
gh_version = (re.search(r'NOTEBOOK_VERSION = \\"([^\\"]+)\\"', nb.stdout) or [None, None])[1] if nb.returncode == 0 else None
after_pin = git("diff", "--name-only", PIN_SHA, remote_head, cwd=AGENT_ROOT).splitlines() if remote_head != PIN_SHA else []
SYNC = {"notebook_version": NOTEBOOK_VERSION, "notebook_version_on_github_head": gh_version, "github_remote_head": remote_head,
        "checkout_head": head, "pinned_sha": PIN_SHA, "pin_on_branch": on_branch, "working_tree_clean": status == "",
        "files_changed_after_pin": after_pin}
print(json.dumps(SYNC, indent=2))
assert head == PIN_SHA, "STOP: checkout HEAD ≠ pin（不自動 checkout/reset；請換 AGENT_ROOT 或重啟 runtime）"
assert status == "", "STOP: checkout 有變更"
assert on_branch, "STOP: pin 不在 GitHub 分支上"
assert gh_version in (None, NOTEBOOK_VERSION), f"STOP: 這是舊 notebook（{NOTEBOOK_VERSION}），GitHub 上是 {gh_version}；請從 GitHub 重新開啟"
code_after_pin = [p for p in after_pin if not p.startswith(("docs/", "notebooks/"))]
assert not code_after_pin, f"STOP: pin 之後 GitHub 上有程式／治理變更 {code_after_pin}；需要新的 pin"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(AGENT_ROOT) + "[test]"], check=True)
print("VERSION SYNC: PASS")

In [ ]:
# --- 3. 共用 Schema 契約檢查 + Phase 5-A3 測試（subprocess；Core 不安裝）---
def py(code_or_args, **kw):
    args = ["-c", code_or_args] if isinstance(code_or_args, str) else code_or_args
    return subprocess.run([sys.executable, *args], cwd=AGENT_ROOT, text=True, capture_output=True, **kw)
r = py("import json;from strategy_agent.phase5a import contract_check as c;print(json.dumps(c()))")
assert r.returncode == 0, r.stderr[-2000:]
CONTRACT = json.loads(r.stdout.strip().splitlines()[-1])
print(json.dumps(CONTRACT, indent=2))
assert CONTRACT["status"] == "PASS", "STOP: shared schema drift"
t = py(["-m", "pytest", "-q", "tests/test_phase5a3_integration.py", "tests/test_orchestrator_lifecycle.py",
        "tests/test_orchestrator_request_contract.py", "tests/test_proposal_agent.py", "tests/test_proposal_contract.py"], timeout=1800)
print(t.stdout[-1500:]); assert t.returncode == 0, "STOP: tests failed"
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: tests modified the checkout"

In [ ]:
# --- 4. Drive（只讀檢查既有 Archive；不可用時雲端驗收 = PENDING）---
CLOUD = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CLOUD = Path(DRIVE_BASE).is_dir()
except Exception as e:                       # noqa: BLE001
    print("Drive unavailable:", type(e).__name__, e)
SRC = Path(DRIVE_BASE) / REAL_RUN
ARCHIVE_OK = CLOUD and SRC.is_dir() and hashlib.sha256((SRC / "MANIFEST.json").read_bytes()).hexdigest() == REAL_ARCHIVE_MANIFEST_SHA256
print({"cloud": CLOUD, "real_archive_present_and_pinned": ARCHIVE_OK, "files": sorted(p.name for p in SRC.iterdir()) if ARCHIVE_OK else None})
OUT_BASE = Path(DRIVE_BASE) / "integrations" if CLOUD else Path("/content/integrations_LOCAL_CLOUD_PENDING")
OUT = OUT_BASE / f"{datetime.now(ZoneInfo('Asia/Taipei')):%Y%m%d_%H%M%S}_{PIN_SHA[:7]}"
assert not OUT.exists(), "STOP: output exists (never overwritten)"
print("Output:", OUT)

In [ ]:
# --- 5. Route B 核准（必須由研究負責人本人填寫；留空 = PENDING_USER_APPROVAL，不重播）---
# APPROVER：你在 governance/approvers.json 的 research_leads 身分；APPROVAL_REF：你本人的 https 核准紀錄（例如 PR 留言連結）。
# 注意：Orchestrator 只比對 allow-list，這不是經過驗證的身分（Phase 5-B 前置工作）。
APPROVER = ""
APPROVAL_REF = ""
REAL = bool(ARCHIVE_OK and APPROVER and APPROVAL_REF.startswith("https://"))
print("Route B real replay:", "WILL RUN (read-only, 0 batches)" if REAL else "NOT RUN → PENDING")

In [ ]:
# --- 6. A3-01..A3-10 acceptance（全新 subprocess：A3-10 的 executor-module 檢查才有意義）---
cmd = ["-m", "strategy_agent.phase5a", "acceptance", "--out-dir", str(OUT)] + (["--cloud"] if CLOUD else [])
if REAL:
    cmd += ["--real-archive-root", DRIVE_BASE, "--approver", APPROVER, "--approval-ref", APPROVAL_REF]
r = py(cmd, timeout=3600)
print(r.stdout[-6000:]); print(r.stderr[-3000:])
RES = json.loads(r.stdout)
print("VERDICT:", RES["verdict"])
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: acceptance modified the checkout"

In [ ]:
# --- 7. Immutable export + 下載 ---
import zipfile
names = sorted(p.name for p in OUT.iterdir() if p.is_file()); print(names)
(OUT / "notebook_context.json").open("x").write(json.dumps({"notebook": NOTEBOOK_VERSION, "version_sync": SYNC,
    "clean_colab_runtime": CLEAN_RUNTIME, "cloud": CLOUD, "real_replay_requested": REAL,
    "python": sys.version.split()[0]}, indent=2))
print((OUT / "run_info.txt").read_text())
zp = Path("/content") / (OUT.name + "_phase5a3_integration.zip")
with zipfile.ZipFile(zp, "w") as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p, p.name)
try:
    from google.colab import files; files.download(str(zp))
except Exception as e:                       # noqa: BLE001
    print("download skipped:", e, "| zip:", zp)